# 02. 크롤링 파이프라인 — 데이터 확인 및 Logic App 수동 실행

이 노트북에서는:
1. Blob Storage에 크롤링된 파일이 몇개인지 확인합니다.
2. 만약 아무 데이터도 없다면 Logic App을 수동으로 실행합니다.
3. Logic App 상태와 최근 실행 이력을 확인합니다.


## 1. 환경 설정

In [16]:
import os
import sys
import json
import subprocess
from datetime import datetime
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

CRAWL_URL    = os.environ.get('AZURE_FUNCTION_CRAWL_URL', '')
STORAGE_NAME = os.environ.get('AZURE_STORAGE_ACCOUNT_NAME', '')
CONTAINER    = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
RG_NAME      = os.environ.get('AZURE_RESOURCE_GROUP', 'rg-rag-indexing-lab-swc')
LOGIC_APP    = os.environ.get('AZURE_LOGIC_APP_NAME', '')

print(f"Function URL : {CRAWL_URL}")
print(f"Storage      : {STORAGE_NAME}")
print(f"Container    : {CONTAINER}")
print(f"Logic App    : {LOGIC_APP}")

Function URL : https://func-crawl-ragi-dyn6dtfu.azurewebsites.net/api/crawl
Storage      : stragidyn6dtfun6
Container    : raw-documents
Logic App    : logic-crawl-ragi-dyn6dtfu


## 2. Blob Storage 크롤링 결과 확인

현재 저장된 크롤링 데이터의 파일 개수를 확인합니다.

In [17]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()
blob_svc = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)
container_client = blob_svc.get_container_client(CONTAINER)

blobs = list(container_client.list_blobs())

# Blob 경로: {source}/{date}/{source}_{seq}.json
sources = sorted(set(b.name.split('/')[0] for b in blobs if '/' in b.name))

print(f"\n{'='*60}")
print(f"크롤링 데이터 현황")
print(f"{'='*60}")
print(f"총 {len(blobs):,}개 파일 | {len(sources)}개 소스 폴더\n")

if len(blobs) == 0:
    print("⚠️  아직 크롤링된 데이터가 없습니다.")
    print("다음 단계에서 Logic App을 수동으로 실행하세요.\n")
else:
    for src in sources:
        src_blobs = [b for b in blobs if b.name.startswith(f"{src}/")]
        dates = sorted(set(b.name.split('/')[1] for b in src_blobs if b.name.count('/') >= 2), reverse=True)
        print(f"  /{src}/")
        print(f"    전체: {len(src_blobs)}개 파일, {len(dates)}개 날짜")
        for d in dates[:3]:
            cnt = sum(1 for b in src_blobs if f"{src}/{d}/" in b.name)
            print(f"      {d}: {cnt}개")
        print()


크롤링 데이터 현황
총 59,551개 파일 | 5개 소스 폴더

  /_logs/
    전체: 179개 파일, 2개 날짜
      2026-05-13: 23개
      2026-05-12: 156개

  /admrul/
    전체: 6831개 파일, 2개 날짜
      2026-05-13: 3282개
      2026-05-12: 3549개

  /detc/
    전체: 24985개 파일, 2개 날짜
      2026-05-13: 12004개
      2026-05-12: 12981개

  /expc/
    전체: 8715개 파일, 1개 날짜
      2026-05-12: 8715개

  /prec/
    전체: 18841개 파일, 2개 날짜
      2026-05-13: 9573개
      2026-05-12: 9268개



In [18]:
# 첫 번째 데이터 소스의 최신 JSON 파일 미리보기 (_logs 등 메타 폴더 제외)
data_sources = [s for s in sources if not s.startswith('_')]
if data_sources:
    src = data_sources[0]
    json_blobs = sorted(
        [b for b in blobs if b.name.startswith(f"{src}/") and b.name.endswith('.json')],
        key=lambda b: b.name,
        reverse=True,
    )
    if json_blobs:
        blob = container_client.get_blob_client(json_blobs[0].name)
        content = blob.download_blob().readall().decode('utf-8')
        doc = json.loads(content)
        print(f"\n[샘플] {json_blobs[0].name}\n")
        pretty = json.dumps(doc, ensure_ascii=False, indent=2)
        print(pretty[:1500])
        if len(pretty) > 1500:
            print("\n... (생략)")
    else:
        print(f"\n/{src}/ 폴더에 .json 파일 없음")
else:
    print("\n데이터 소스 폴더가 없습니다.")


[샘플] admrul/2026-05-13/admrul_9992.json

{
  "seq": "9992",
  "source": "행정심판재결례",
  "사건명": "도시계획시설변경결정처분취소청구",
  "사건번호": "1996-01844",
  "재결일자": "1997-01-31T00:00:00Z",
  "재결기관": "국민권익위원회",
  "재결결과": "기각",
  "재결요지": "사 건 96-1844 도시계획시설변경결정처분취소등청구 청 구 인 이 ○ ○ 서울특별시 ○○구 ○○동 562의 14 대리인 변호사 김 ○ ○ 피청구인 경기도지사 청구인이 1996. 8. 22. 제기한 심판청구에 대하여 1997년도 제3회 국무총리행정심판위원회는 주문과 같이 의결한다.",
  "주문": "1. 청구인의 도시계획시설변경결정처분취소청구를 각하한다. 2. 청구인의 도시계획시설변경결정처분무효확인청구을 기각한다.",
  "청구취지": "피청구인이 1993. 9. 16. 청구인의 토지에 대하여 한 도시계획시설변경결정처분은 이를 취소하거나 무효임을 확인한다.",
  "이유": "1. 사건개요 피청구인이 1993. 9. 16. 경기도 고시 제1993-293호로 ○○도시계획(재정비)변경결정을 하면서 청구인 소유의 토지 경기도 ○○시 ○○읍 ○○리 1027의10번지 소재 전 1,636제곱미터를 도로 및 하천시설로 결정하였다. 2. 청구인 주장 이에 대하여 청구인은, 피청구인이 1993. 9. 16. 경기도고시 제1993-293호로 청구인 소유의 경기도 ○○시 ○○읍 ○○리 1027의10번지 소재 전 1,636제곱미터의 토지일대의 ○○ 도시계획(재정비)변경결정을 하면서 위 토지에 관하여 도로 및 하천시설결정을 하였으나, 도시기본계획 및 도시계획변경의 입안자인 청구외 ○○시장은 도시계획변경을 입안함에 있어서 주민들의 의견청취절차를 전혀 거치지 아니하는 잘못을 범하였고, 위 토지는 일년내내 물이 흐르는 적이 없는 등 하천시설로 이용할 하등의 필요성이 없는 곳이므로 위 도시계획시설변경결정

## 3. Logic App 수동 실행 (Durable orchestration)

아래 셀을 실행하면 Logic App workflow `logic-crawl-ragi-*`의 트리거 `Daily_Schedule_Crawl_and_Preprocess`가 즉시 호출됩니다.

**파이프라인 흐름 (Method B — Flex Consumption FC1 + 단일 Durable orchestrator):**

1. Logic App → Durable Functions HTTP Starter (`POST /api/orchestrators/crawl_preprocess`, `operationOptions: DisableAsyncPattern` 으로 즉시 `statusQueryGetUri` 수신)
2. `crawl_preprocess_orchestrator` (단일 flat orchestrator):
   - **Step 1** — 4개 source × `LIST_PAGE_CHUNK` (=30 페이지) wave 단위로 `activity_list_seqs` 병렬 fan-out (소스당 최대 `LIST_MAX_WAVES`=20 wave)
   - **Step 2** — 모든 source × 모든 batch (50 seq/배치) `activity_crawl_detail_batch` 병렬 fan-out
   - **Step 3** — 4개 source `activity_preprocess_source` 병렬 fan-out
3. Logic App `Until_Completed` 루프가 30초 간격으로 status URL 폴링, `runtimeStatus ∈ {Completed, Failed, Terminated, Canceled}` 시 종료

**보장 사항:**

1. **중복 방지** — listing 단계에서 기존 Blob seq 제외 + 목록 dedup + Blob 업로드 `overwrite=False` + preprocess 가 기존 part 삭제 후 재생성 (idempotent)
2. **순서 보장** — orchestrator 가 step 1 → 2 → 3 순으로 활동 완료 후에만 다음 단계 호출 (`yield context.task_all(...)` 동기화)
3. **고병렬** — `detail_workers=20` (배치 내 동시 HTTP 요청) × Flex Consumption 자동 스케일 (최대 40 인스턴스) × source × wave/batch fan-out
4. **Bicep 인프라** — FC1 Function (`infra/{sweden,korea}/modules/function-crawler-consumption.bicep`) + Logic App (`logic-app-crawl.bicep`) + 패치 (`infra/sweden/patches/add-method-b.bicep`) 전체 코드화


In [20]:
import shutil
import time

def run_az(args):
    """Windows/Linux 모두에서 az CLI 호출 (Windows에서 az는 .cmd 배치 파일)."""
    az_path = shutil.which("az") or "az"
    return subprocess.run([az_path] + args, capture_output=True, text=True)

# 1) Workflow 이름 확정 (.env 우선, 없으면 RG에서 자동 검색)
workflow_name = LOGIC_APP.strip() if LOGIC_APP else ""
if not workflow_name:
    res = run_az([
        "logic", "workflow", "list",
        "--resource-group", RG_NAME,
        "--query", "[?contains(name, 'logic-crawl')].name",
        "-o", "tsv",
    ])
    names = [n.strip() for n in res.stdout.splitlines() if n.strip()]
    if names:
        workflow_name = names[0]
print(f"Logic App: {workflow_name}")

# 2) Subscription ID 조회
sub_res = run_az(["account", "show", "--query", "id", "-o", "tsv"])
sub_id = sub_res.stdout.strip()

# 3) 트리거 목록 조회 (REST — `az logic workflow trigger` subcommand 가 deprecate 됨)
trig_url = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{RG_NAME}"
    f"/providers/Microsoft.Logic/workflows/{workflow_name}/triggers"
    f"?api-version=2019-05-01"
)
trig_res = run_az(["rest", "--method", "get", "--url", trig_url, "--query", "value[].name", "-o", "tsv"])
triggers = [t.strip() for t in trig_res.stdout.splitlines() if t.strip()]
print(f"트리거: {triggers}")
trigger_name = triggers[0] if triggers else "Daily_Schedule_Crawl_and_Preprocess"

# 4) 트리거 즉시 실행 (REST POST)
print(f"\n▶ Logic App workflow 실행 중...")
run_url = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{RG_NAME}"
    f"/providers/Microsoft.Logic/workflows/{workflow_name}/triggers/{trigger_name}/run"
    f"?api-version=2019-05-01"
)
run_res = run_az(["rest", "--method", "post", "--url", run_url])
if run_res.returncode == 0:
    print(f"✅ 트리거 호출 성공 — 백그라운드에서 Durable orchestration 진행 중")
    print(f"   진행 상황은 Azure Portal Logic App Run history 또는 아래 셀에서 확인하세요.")
else:
    print(f"❌ 트리거 호출 실패:\n{run_res.stderr}")

# 5) 5초 대기 후 최신 run 상태 1회 표시
time.sleep(5)
url = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{RG_NAME}"
    f"/providers/Microsoft.Logic/workflows/{workflow_name}/runs"
    f"?api-version=2019-05-01&$top=1"
)
latest = run_az(["rest", "--method", "get", "--url", url, "-o", "json"])
if latest.returncode == 0 and latest.stdout.strip():
    runs = json.loads(latest.stdout).get("value", [])
    if runs:
        r = runs[0]
        print(f"\n최신 run:")
        print(f"  status   : {r['properties'].get('status')}")
        print(f"  startTime: {r['properties'].get('startTime')}")
        print(f"  runId    : {r['name']}")


Logic App: logic-crawl-ragi-dyn6dtfu
트리거: ['Daily_Schedule_Crawl_and_Preprocess']

▶ Logic App workflow 실행 중...
✅ 트리거 호출 성공 — 백그라운드에서 Durable orchestration 진행 중
   진행 상황은 Azure Portal Logic App Run history 또는 아래 셀에서 확인하세요.


## 4. Logic App 상태 확인

In [21]:
import shutil

def run_az(args):
    """Windows/Linux 모두에서 az CLI 호출 (az는 Windows에서 .cmd 배치 파일)."""
    az_path = shutil.which("az") or "az"
    return subprocess.run([az_path] + args, capture_output=True, text=True)

workflow_name = LOGIC_APP.strip() if LOGIC_APP else ""
if not workflow_name:
    discover_res = run_az([
        "logic", "workflow", "list",
        "--resource-group", RG_NAME,
        "--query", "[].name",
        "-o", "tsv",
    ])
    if discover_res.returncode == 0:
        names = [n.strip() for n in discover_res.stdout.splitlines() if n.strip()]
        if len(names) == 1:
            workflow_name = names[0]
            print(f"\n자동 선택: {workflow_name}\n")
        elif len(names) > 1:
            print(f"\n여러 워크플로우 발견: {names}")
            print(".env에서 AZURE_LOGIC_APP_NAME을 명시하세요.\n")
            workflow_name = None
        else:
            print(f"\n리소스 그룹 '{RG_NAME}'에 Logic App 워크플로우가 없습니다.\n")
            workflow_name = None
    else:
        print(f"az CLI 조회 실패: {discover_res.stderr.strip()}")

if workflow_name:
    show_res = run_az([
        "logic", "workflow", "show",
        "--name", workflow_name,
        "--resource-group", RG_NAME,
        "--query", "{name:name, state:properties.state, location:location}",
        "-o", "json",
    ])
    if show_res.returncode == 0 and show_res.stdout.strip():
        print(json.dumps(json.loads(show_res.stdout), ensure_ascii=False, indent=2))
    else:
        print(f"조회 실패: {show_res.stderr.strip()}")

{
  "location": "swedencentral",
  "name": "logic-crawl-ragi-dyn6dtfu",
  "state": null
}


## 5. 최근 실행 이력

Logic App의 최근 실행 상태와 실패 건수를 확인합니다.

In [22]:
if workflow_name:
    sub_res = run_az(["account", "show", "--query", "id", "-o", "tsv"])
    if sub_res.returncode == 0 and sub_res.stdout.strip():
        sub_id = sub_res.stdout.strip()
        url = (
            f"https://management.azure.com/subscriptions/{sub_id}"
            f"/resourceGroups/{RG_NAME}"
            f"/providers/Microsoft.Logic/workflows/{workflow_name}/runs"
            f"?api-version=2019-05-01"
        )
        run_res = run_az([
            "rest",
            "--method", "get",
            "--url", url,
            "--query", "{total:length(value), succeeded:length(value[?properties.status=='Succeeded']), failed:length(value[?properties.status=='Failed']), latest_status:value[0].properties.status, latest_time:value[0].properties.startTime}",
            "-o", "json",
        ])
        if run_res.returncode == 0:
            data = json.loads(run_res.stdout)
            total = data.get("total", 0)
            succeeded = data.get("succeeded", 0)
            failed = data.get("failed", 0)
            latest_status = data.get("latest_status", "Unknown")
            latest_time = data.get("latest_time", "N/A")

            print(f"\n{'='*60}")
            print(f"Logic App 실행 이력")
            print(f"{'='*60}")
            print(f"총 실행   : {total}건")
            print(f"성공     : {succeeded}건")
            print(f"실패     : {failed}건")
            print(f"최근 상태: {latest_status}")
            print(f"최근 실행: {latest_time}")
            print()
            if failed > 0:
                print(f"❌ 최근 실행 중 {failed}건 실패가 감지되었습니다.")
                print("   Azure Portal에서 실행 이력 상세를 확인하세요.")
            else:
                print("✅ 최근 실행에서 실패가 감지되지 않았습니다.")
        else:
            print("실행 이력 조회 실패 (권한 확인 필요)")


Logic App 실행 이력
총 실행   : 8건
성공     : 6건
실패     : 0건
최근 상태: Succeeded
최근 실행: 2026-05-13T15:24:47.6578245Z

✅ 최근 실행에서 실패가 감지되지 않았습니다.
